# MorphoNAS-playbook — editing the source (Tier 3)

The main hands-on notebook `pip install`s the playbook as a **read-only snapshot**,
so you can only change *parameters*. This notebook instead **clones the repo and
installs it editable**, so every file — including the vendored MorphoNAS
**engine** — becomes a real, editable file in Colab. Change a growth rule, the
propagation, a default, anything, and re-run.

Use this when you want to go past parameters and actually modify the code.


In [ ]:
# Clone + editable install
!git clone -q https://github.com/ukma-morphonas-lab/MorphoNAS-playbook.git
!pip install -q -e MorphoNAS-playbook

import os, sys, types, importlib
# An editable install registers via a .pth finder that an ALREADY-RUNNING kernel
# doesn't process, so `import morphonas_playbook` would fail until a restart. Put
# the clone on sys.path now -> the kernel imports the same (editable) source at once.
CLONE = os.path.abspath("MorphoNAS-playbook")
if CLONE not in sys.path:
    sys.path.insert(0, CLONE)

# IPython autoreload on some Colab images imports the removed `imp` module (py3.12);
# shim it, and keep this non-fatal (worst case: edit, then Restart runtime).
if "imp" not in sys.modules:
    try:
        import imp  # noqa: F401
    except ModuleNotFoundError:
        _m = types.ModuleType("imp"); _m.reload = importlib.reload
        sys.modules["imp"] = _m
try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
    print("autoreload: on")
except Exception as e:
    print("autoreload off (", e, ") -- edit, then Restart runtime, or importlib.reload")

import morphonas_playbook as mp
from IPython.display import Image, display
print("imported from:", mp.__file__)   # /content/MorphoNAS-playbook/... (the clone)


## Where the files are

Open the **Files** pane (the folder icon on the left), expand
`MorphoNAS-playbook/morphonas_playbook/`, and double-click any file to edit it
in-browser (Ctrl+S / Cmd+S to save):

- `core.py` — grow / attach / evaluate primitives, the `TASKS` registry
- `growth.py` · `control.py` · `evolve.py` — Parts 1 / 2 / 3-4 helpers
- **`engine/`** — the actual MorphoNAS engine (pinned snapshot): `grid.py`
  (reaction-diffusion + cell division + axon growth), `genome.py`,
  `neural_propagation.py`, `genetic_algorithm.py`, `fitness_functions.py` …

Because the install is *editable*, saving a file is all you need — `%autoreload`
picks it up on the next cell you run.


## Worked example: change the source, see the effect

You'd normally edit in the Files pane, but you can also patch a file from a cell.
Here we inject a print into `grow()` in `core.py`, then call it — the message
proves your edit is live — and then revert.


In [ ]:
# 1) edit the source (a one-line patch into core.grow)
!sed -i 's/^    grid = Grid(genome)/    print("[edited grow] developing a genome..."); grid = Grid(genome)/' \
    MorphoNAS-playbook/morphonas_playbook/core.py

# 2) run it — autoreload reloads the edited module; the print fires
G = mp.graph_of(mp.core.grow(mp.random_genome(39)))
print("grew", mp.n_nodes(G), "neurons")


In [ ]:
# 3) revert the file when you're done experimenting
!cd MorphoNAS-playbook && git checkout -q morphonas_playbook/core.py
print("reverted core.py")


## autoreload vs. restart — and the one gotcha

- **`%autoreload 2`** reloads any changed module before each cell runs, so most
  edits (functions, the `TASKS` table, palette colours, growth constants) take
  effect immediately.
- **Restart the runtime** (Runtime -> Restart session) if autoreload gets
  confused — e.g. after big/structural edits, or edits to values read once at
  import time. A restart + re-run of the setup cell always reflects the latest source.
- **`run_ga` gotcha (multiprocessing):** the GA farms fitness out to worker
  processes that **re-import the editable package fresh**, so they *do* see your
  edits. But the *main* process may hold stale copies until autoreload/restart.
  For a clean run after editing engine code, either restart the runtime, or call
  `run_ga(..., max_workers=1)` so everything runs in one process.


In [ ]:
# quick sanity check that edits flow through end-to-end (engine still works)
G = mp.graph_of(mp.grow(mp.random_genome(39)))
mean, _ = mp.evaluate(G, episodes=5)
print("grow + evaluate OK:", mp.n_nodes(G), "neurons, reward", mean)


## Heads-up: edits live only in this Colab session

The clone lives in the Colab VM's disk (`/content/...`), which is wiped when the
runtime disconnects. To keep changes: download the edited files, save the clone to
Google Drive, or `git commit` + push to your own fork. Nothing here touches the
published repo unless you explicitly push.
